# Lesson 1: What Makes an AI Application an Agent?

## Lesson introduction

_I’ll write this introduction after completing the lesson and reflecting on what I learned._

## Expected data flows

![Three vertical data flows comparing an LLM-only application, a deterministic workflow, and a simple agent](assets/three-patterns-data-flow.svg)

## Test Case 1: LLM Only

In this first test, I’ll give the model the full company dataset in the prompt and ask it to identify the company with the highest ARR per employee. There are no tools and no execution loop—the model has to interpret the data, do the calculation, and return an answer in a single response.

### Setup

1. Install the required packages in the notebook's virtual environment: `%pip install openai python-dotenv`.
2. Open `.env` in this lesson folder.
3. Replace `replace_with_your_openai_api_key` with your own OpenAI API key.
4. Keep `.env` local. Git ignores it, so the key will not be pushed to the repository.

The committed `.env.example` file shows the variable that other learners need to configure without containing a real key.

In [ ]:
# Install these packages once in the notebook's virtual environment:
# %pip install openai python-dotenv

import json
import os
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI


# Find the lesson folder whether Jupyter starts here or at the repository root.
lesson_path = Path(".")
if not (lesson_path / "data/companies.json").exists():
    lesson_path = Path("lesson-01-agent-basics")

# Load the API key from the local .env file.
load_dotenv(lesson_path / ".env")
api_key = os.getenv("OPENAI_API_KEY")
if not api_key or api_key == "replace_with_your_openai_api_key":
    raise ValueError("Add your OpenAI API key to lesson-01-agent-basics/.env")

client = OpenAI(api_key=api_key)

# Load the same local dataset that each test case will use.
data_path = lesson_path / "data/companies.json"
with data_path.open(encoding="utf-8") as data_file:
    companies = json.load(data_file)

# The model receives the question and the complete dataset as text.
question = (
    "Using ARR per employee as the efficiency metric, which company generates "
    "recurring revenue most efficiently? Calculate ARR per employee for every "
    "company, rank the companies from highest to lowest, and identify the leader."
)

prompt = f"""
{question}

ARR is expressed in millions of US dollars. Show the calculation for each company
and round ARR per employee to the nearest dollar. Use only the supplied dataset.

Company dataset:
{json.dumps(companies, indent=2)}
"""

# This is the entire LLM-only application: one request, with no tools or loop.
response = client.responses.create(
    model="gpt-5.4-nano",
    reasoning={"effort": "none"},
    input=prompt,
)

# Render the model's Markdown rather than displaying it as plain text.
display(Markdown(response.output_text))